In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
from one.api import ONE
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainbox.ephys_plots import plot_brain_regions
from iblatlas.atlas import AllenAtlas
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainwidemap.bwm_loading import merge_probes
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.task.trials import find_trial_ids
from brainbox.io.one import SessionLoader
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox.singlecell import bin_spikes2D
import numpy as np
from iblatlas.atlas import BrainRegions
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
import pickle as pkl
from tqdm import tqdm
from pathlib import Path
import warnings
from sklearn.ensemble import RandomForestClassifier
from ibl_info.prepare_data_pid import (
    cleaned_regions_flags,
    get_new_cinc_intervals,
    prepare_ephys_data,
)
from ibl_info.utils import (
    alternate_discretize,
    compute_mutual_information,
    compute_pid,
    compute_trivariate_mi,
    FIRING_RATE,
    discretize,
    discretize_keeping_zeros,
    equipopulated_binning,
)
import os
import concurrent.futures
import functools
import random
from ibl_info.utils import check_config

In [2]:
subject_id = "CSH_ZAD_022"
eid = "a82800ce-f4e3-4464-9b80-4c3d6fade333"
session_id = eid

In [3]:
one = ONE()

In [4]:
trials, mask = load_trials_and_mask(one, session_id, exclude_nochoice=True, exclude_unbiased=True)

In [7]:
## proper code for generating pseudosessions

from brainwidemap.decoding.functions import nulldistributions

In [12]:
metadata = {"subject": subject_id, "eid": eid, "probe_name": "probe00"}

In [13]:
from brainbox.task.closed_loop import generate_pseudo_session

In [15]:
pseudosess = generate_pseudo_session(trials, generate_choices=False)

In [20]:
from brainwidemap.decoding.functions.process_targets import optimal_Bayesian, check_bhv_fit_exists
from behavior_models import models

In [ ]:
my_model = models.ActionKernel(
    path_to_results="results_behavioral", session_uuids= , df_trials=trials, single_zeta=True
)

In [24]:
from behavior_models.utils import format_input as mut_format_input

In [ ]:
stim, _, side = mut_format_input( # pyright: ignore[reportAssignmentType]
    [pseudosess.signed_contrast.values],
    [trials.choice.values],
    [pseudosess.stim_side.values],
)

In [31]:
import torch

In [34]:
my_model.load_or_train(remove_old=False, adaptive=True)

2025-11-12 22:47:53 INFO     base_models.py:109  Launching MCMC procedure with 4 chains, 5000 max steps and [0.05 0.04 0.02 0.02] std_RW. Early stopping is activated
2025-11-12 22:47:53 INFO     base_models.py:119  with adaptive MCMC...
2025-11-12 22:47:53 INFO     base_models.py:121  initial point for MCMC is [[0.5    0.5    0.25   0.25  ]
 [0.75   0.25   0.375  0.125 ]
 [0.25   0.75   0.125  0.375 ]
 [0.375  0.375  0.3125 0.0625]]


 10%|▉         | 498/5000 [00:17<02:40, 28.12it/s]

2025-11-12 22:48:11 INFO     base_models.py:165  Adaptive MCMC starting...


 12%|█▏        | 600/5000 [00:21<02:34, 28.43it/s]

2025-11-12 22:48:15 INFO     base_models.py:189  acceptance is 0.22874999999999998


...

 18%|█▊        | 900/5000 [00:32<02:27, 27.89it/s]

2025-11-12 22:48:26 INFO     base_models.py:189  acceptance is 0.22805555555555554


 20%|█▉        | 999/5000 [00:35<02:25, 27.47it/s]

2025-11-12 22:48:29 INFO     base_models.py:189  acceptance is 0.229
2025-11-12 22:48:29 INFO     base_models.py:159  Early stopping criteria was validated at step 1001. R values are: [1.02902988 1.01909696 0.99857596 1.01235494]


2025-11-12 22:48:29 INFO     base_models.py:194  final posterior_mean is [0.10474322 0.1293245  0.04355325 0.0632285 ]
2025-11-12 22:48:29 INFO     base_models.py:203  acceptance ratio is of 0.22952047952047955. Careful, this ratio should be close to 0.234. If not, change the standard deviation of the random walk
2025-11-12 22:48:29 INFO     base_models.py:312  results of inference SAVED in results_behavioral/model_actKernel_single_zeta/train_a82800ce.pkl


In [35]:
arr_params = my_model.get_parameters(parameter_type="posterior_mean")[None]

In [37]:
valid = np.ones([1, pseudosess.index.size], dtype=bool)

In [ ]:
stim.shape

(1, 4)

In [51]:
act_sim, stim, side = my_model.simulate(  # type: ignore
    arr_params, stim[0, :], side[0, :], nb_simul=1, only_perf=False, return_prior=False
)

<function brainwidemap.bwm_loading.bwm_units(one=None, freeze='2023_12_bwm_release', rt_range=(0.08, 0.2), min_errors=3, min_qc=1.0, min_units_sessions=(5, 2), enforce_version=True)>